# Notes
Auto read `data_input.json` and then predict IC20s and hazard levels for them.

The default output will be written to `<working dir>/data_output.csv`

In [ ]:
# Public methods
import os, json, tqdm

import numpy as np
import pandas as pd

def join(*path) -> str:
    return os.path.join(WORKING_DIR, *path)

WORKING_DIR = "Viability"


# Predict
We have **binary classification models** (indicate $\mathrm{active}$ or $\mathrm{inactive}$) and **regression models** (predict $IC_\mathrm{20,free}$ and $IC_\mathrm{20,cell}$)  
Each type of model has 3 endpoints (**viability**/**apoptosis**/**mitochondrial toxicity**)

In [ ]:
# Estimator
from typing import Literal

from tabpfn import load_fitted_tabpfn_model


class BaseEstimator():
    def __init__(
            self, 
            date: str,
            device: str,
            model: Literal['xgb','tabpfn'],
            model_type: Literal['classifier', 'regressor'],
            run_num: int,
            features: list,
            cat_dict: dict = {},
            cat_keys: list[str] = [],
    ):
        """
        Ensemble model

        :param device: refer to `torch.device`
        :param model: tabpfn/xgb
        :param model_type: classifier/regressor
        :param features: features
        :param date: `metadata['date']`
        :param run_num: `metadata['run_num']`
        :param cat_dict: `metadata['cat_dict']`
        :param cat_keys: `metadata['cat_keys']`
        """
        self.device = device
        self.model_type = model_type
        self.model = model
        self.run_num = run_num
        self.cat_dict = cat_dict
        self.cat_keys = cat_keys
        self.features = features
        self.date = date

        # Load models
        self.model_list = []
        for idx in range(self.run_num):
            if self.model == "xgb":
                # XGBoost
                path = join("model", f"{self.date}_{idx}.xgb_fit")
                self.model_list.append(path)
            elif self.model == "tabpfn":
                # TabPFN
                path = join("model", f"{self.date}_{idx}.tabpfn_fit")
                self.model_list.append(path)
            else:
                raise ValueError("unknown model")

    def _load_model(self, model_path:str):
        'load models'
        if self.model == "tabpfn":
            # TabPFN
            model = load_fitted_tabpfn_model(model_path, device=self.device)
        elif self.model == "xgb":
            pass

        return model


class ViabilityEstimator(BaseEstimator):
    'for Viability'

    def predict_proba(self, X:pd.DataFrame):
        'predict(for classification only)'
        assert self.model_type == 'classifier'
        input_X = X[self.features]

        # 3-d array(models*samples*categories)
        shape = (self.run_num, input_X.shape[0], len(self.cat_keys))
        proba_array = np.zeros(shape, dtype=np.float64)

        # get outputs for each model
        for idx, model_path in enumerate(self.model_list):
            model = self._load_model(model_path)
            proba = model.predict_proba(input_X)
            proba_array[idx] = np.asarray(proba)

        # merge
        proba_mean = np.mean(proba_array, axis=0)
        return proba_mean

    def predict(self, X:pd.DataFrame):
        'predict'
        input_X = X[self.features]

        # 2-d array(models*samples*categories)
        shape = (self.run_num, input_X.shape[0])
        pred_array = np.zeros(shape, dtype=np.float64)

        # get outputs for each model
        for idx, model_path in enumerate(self.model_list):
            model = self._load_model(model_path)
            pred = model.predict(input_X)
            pred_array[idx] = pred

        # merge
        pred_all_models = np.vstack(pred_array)
        pred_mean = np.mean(pred_all_models, axis=0)

        if self.model_type == "classifier":
            return np.round(pred_mean).astype(np.int64)
        else:
            return pred_mean


class PotencyEstimator(BaseEstimator):
    'for potency (DEPRECATED)'
    def predict_proba(self, X:pd.DataFrame):
        'predict(for classification)'
        assert self.model_type == 'classifier'
        assert len(self.features) == self.run_num

        # 3-d array(models*samples*categories)
        shape = (self.run_num, X.shape[0], len(self.cat_keys))
        proba_array = np.zeros(shape, dtype=np.float64)

        for idx, model_path in enumerate(self.model_list):
            # for potency, every model uses different features
            features = self.features[idx]
            input_X = X[features]

            # load model and predict
            model = self._load_model(model_path)
            proba = model.predict_proba(input_X)
            proba_array[idx] = np.asarray(proba)

        # merge
        proba_mean = np.mean(proba_array, axis=0)
        return proba_mean


In [ ]:
# Define running parameters
DEVICE = "cuda:1"

# input data
input_data = pd.read_json(join("data_input.json"))

# cat_keys and dict to be defined
cat_keys_a = ['active', 'inactive']
cat_dict_a = {i:idx for idx, i in enumerate(cat_keys_a)}

# the mapping between uuids and endpoints
# you can refer to this dict to enable or disable the corresponding estimators below
MODEL_MAPPINGS = {
    '5b6d43d6-2695-11f1-80cd-60cf848a7bd6': 'Hazard_viability',
    '2c71181e-2776-11f1-9087-60cf848a7bd6': 'IC20_free_viability',
    '6263e416-277a-11f1-9087-60cf848a7bd6': 'IC20_cell_viability',
    '1e7cd41c-2458-11f1-8521-60cf848a7bd6': 'Hazard_mitochondrial',
    'eff9653e-277b-11f1-9087-60cf848a7bd6': 'IC20_free_mitochondrial',
    '53a80028-277b-11f1-9087-60cf848a7bd6': 'IC20_cell_mitochondrial',
    'e7a86ad6-2458-11f1-8521-60cf848a7bd6': 'Hazard_apoptosis',
    '108f5a82-277d-11f1-9087-60cf848a7bd6': 'IC20_free_apoptosis',
    '7c617e02-277d-11f1-9087-60cf848a7bd6': 'IC20_cell_apoptosis',
}

# define models, if some endpoints are not needed, just comment them out
estimator_list = [
    ViabilityEstimator( 
        date='5b6d43d6-2695-11f1-80cd-60cf848a7bd6',
        device=DEVICE,
        model='tabpfn',
        model_type='classifier',
        cat_dict=cat_dict_a,
        cat_keys=cat_keys_a,
        run_num=25,
        features=['rd_MaxAbsEStateIndex', 'rd_MinAbsEStateIndex', 'rd_SPS', 'rd_MolWt', 'rd_NumValenceElectrons', 'rd_MinPartialCharge', 'rd_MaxAbsPartialCharge', 'rd_FpDensityMorgan1', 'rd_FpDensityMorgan2', 'rd_FpDensityMorgan3', 'rd_BCUT2D_MWHI', 'rd_BCUT2D_MWLOW', 'rd_BCUT2D_CHGHI', 'rd_BCUT2D_CHGLO', 'rd_BCUT2D_LOGPHI', 'rd_BCUT2D_LOGPLOW', 'rd_BCUT2D_MRHI', 'rd_AvgIpc', 'rd_BalabanJ', 'rd_BertzCT', 'rd_Chi1v', 'rd_Chi2n', 'rd_Chi3v', 'rd_Chi4n', 'rd_HallKierAlpha', 'rd_Ipc', 'rd_Kappa2', 'rd_Kappa3', 'rd_PEOE_VSA1', 'rd_PEOE_VSA10', 'rd_PEOE_VSA11', 'rd_PEOE_VSA3', 'rd_PEOE_VSA6', 'rd_PEOE_VSA7', 'rd_PEOE_VSA8', 'rd_PEOE_VSA9', 'rd_SMR_VSA1', 'rd_SMR_VSA10', 'rd_SMR_VSA3', 'rd_SMR_VSA5', 'rd_SMR_VSA6', 'rd_SMR_VSA7', 'rd_SMR_VSA9', 'rd_SlogP_VSA1', 'rd_SlogP_VSA10', 'rd_SlogP_VSA11', 'rd_SlogP_VSA12', 'rd_SlogP_VSA2', 'rd_SlogP_VSA3', 'rd_SlogP_VSA4', 'rd_SlogP_VSA5', 'rd_SlogP_VSA7', 'rd_SlogP_VSA8', 'rd_EState_VSA10', 'rd_EState_VSA2', 'rd_EState_VSA3', 'rd_EState_VSA4', 'rd_EState_VSA5', 'rd_EState_VSA6', 'rd_EState_VSA7', 'rd_EState_VSA8', 'rd_VSA_EState1', 'rd_VSA_EState10', 'rd_VSA_EState3', 'rd_VSA_EState4', 'rd_VSA_EState5', 'rd_VSA_EState6', 'rd_VSA_EState7', 'rd_VSA_EState8', 'rd_VSA_EState9', 'rd_FractionCSP3', 'rd_NOCount', 'rd_NumAliphaticCarbocycles', 'rd_NumAliphaticHeterocycles', 'rd_NumAliphaticRings', 'rd_NumAromaticCarbocycles', 'rd_NumAromaticHeterocycles', 'rd_NumAromaticRings', 'rd_NumAtomStereoCenters', 'rd_NumHAcceptors', 'rd_NumHeteroatoms', 'rd_NumHeterocycles', 'rd_NumRotatableBonds', 'rd_NumSaturatedHeterocycles', 'rd_NumSaturatedRings', 'rd_NumSpiroAtoms', 'rd_RingCount', 'rd_MolLogP', 'rd_fr_Al_COO', 'rd_fr_Ar_N', 'rd_fr_Ar_OH', 'rd_fr_COO', 'rd_fr_NH0', 'rd_fr_NH2', 'rd_fr_Ndealkylation2', 'rd_fr_alkyl_halide', 'rd_fr_allylic_oxid', 'rd_fr_aniline', 'rd_fr_aryl_methyl', 'rd_fr_azo', 'rd_fr_bicyclic', 'rd_fr_dihydropyridine', 'rd_fr_halogen', 'rd_fr_imidazole', 'rd_fr_para_hydroxylation', 'rd_fr_piperdine', 'rd_fr_piperzine']
    ),
    ViabilityEstimator( 
        date='1e7cd41c-2458-11f1-8521-60cf848a7bd6',
        device=DEVICE,
        model='tabpfn',
        model_type='classifier',
        cat_dict=cat_dict_a,
        cat_keys=cat_keys_a,
        run_num=15,
        features=["rd_MaxAbsEStateIndex", "rd_MinAbsEStateIndex", "rd_qed", "rd_MolWt", "rd_MinPartialCharge", "rd_MaxAbsPartialCharge", "rd_FpDensityMorgan1", "rd_FpDensityMorgan2", "rd_FpDensityMorgan3", "rd_BCUT2D_MWHI", "rd_BCUT2D_MWLOW", "rd_BCUT2D_CHGHI", "rd_BCUT2D_CHGLO", "rd_BCUT2D_LOGPHI", "rd_BCUT2D_MRHI", "rd_AvgIpc", "rd_BalabanJ", "rd_BertzCT", "rd_Chi0n", "rd_Chi2v", "rd_Chi3n", "rd_Chi4v", "rd_HallKierAlpha", "rd_Ipc", "rd_Kappa2", "rd_PEOE_VSA10", "rd_PEOE_VSA11", "rd_PEOE_VSA6", "rd_PEOE_VSA7", "rd_PEOE_VSA8", "rd_PEOE_VSA9", "rd_SMR_VSA10", "rd_SMR_VSA3", "rd_SMR_VSA6", "rd_SMR_VSA7", "rd_SMR_VSA9", "rd_SlogP_VSA1", "rd_SlogP_VSA10", "rd_SlogP_VSA11", "rd_SlogP_VSA12", "rd_SlogP_VSA2", "rd_SlogP_VSA4", "rd_SlogP_VSA5", "rd_SlogP_VSA7", "rd_SlogP_VSA8", "rd_EState_VSA10", "rd_EState_VSA2", "rd_EState_VSA3", "rd_EState_VSA4", "rd_EState_VSA5", "rd_EState_VSA6", "rd_EState_VSA7", "rd_VSA_EState10", "rd_VSA_EState3", "rd_VSA_EState4", "rd_VSA_EState5", "rd_VSA_EState6", "rd_FractionCSP3", "rd_NumAliphaticCarbocycles", "rd_NumAliphaticHeterocycles", "rd_NumAromaticCarbocycles", "rd_NumAromaticHeterocycles", "rd_NumAromaticRings", "rd_NumAtomStereoCenters", "rd_NumHAcceptors", "rd_NumHeteroatoms", "rd_NumSaturatedCarbocycles", "rd_NumSaturatedHeterocycles", "rd_NumSaturatedRings", "rd_RingCount", "rd_MolLogP", "rd_fr_Al_COO", "rd_fr_Al_OH", "rd_fr_Ar_OH", "rd_fr_COO", "rd_fr_C_O", "rd_fr_Ndealkylation1", "rd_fr_Ndealkylation2", "rd_fr_aniline", "rd_fr_aryl_methyl", "rd_fr_bicyclic", "rd_fr_dihydropyridine", "rd_fr_halogen", "rd_fr_lactam", "rd_fr_nitro", "rd_fr_nitro_arom", "rd_fr_nitro_arom_nonortho", "rd_fr_para_hydroxylation", "rd_fr_piperdine", "rd_fr_thiazole"],
    ),
    ViabilityEstimator( 
        date='e7a86ad6-2458-11f1-8521-60cf848a7bd6',
        device=DEVICE,
        model='tabpfn',
        model_type='classifier',
        cat_dict=cat_dict_a,
        cat_keys=cat_keys_a,
        run_num=75,
        features=["rd_SPS", "rd_MolWt", "rd_NumValenceElectrons", "rd_MaxPartialCharge", "rd_MinAbsPartialCharge", "rd_FpDensityMorgan1", "rd_FpDensityMorgan2", "rd_FpDensityMorgan3", "rd_BCUT2D_MWLOW", "rd_BCUT2D_CHGHI", "rd_BCUT2D_CHGLO", "rd_BCUT2D_LOGPHI", "rd_BCUT2D_LOGPLOW", "rd_AvgIpc", "rd_BalabanJ", "rd_BertzCT", "rd_Chi1v", "rd_Chi2n", "rd_Chi3v", "rd_Chi4n", "rd_HallKierAlpha", "rd_Ipc", "rd_Kappa2", "rd_Kappa3", "rd_PEOE_VSA1", "rd_PEOE_VSA10", "rd_PEOE_VSA6", "rd_PEOE_VSA7", "rd_PEOE_VSA8", "rd_PEOE_VSA9", "rd_SMR_VSA10", "rd_SMR_VSA3", "rd_SMR_VSA5", "rd_SMR_VSA6", "rd_SMR_VSA7", "rd_SMR_VSA9", "rd_SlogP_VSA11", "rd_SlogP_VSA2", "rd_SlogP_VSA5", "rd_SlogP_VSA7", "rd_SlogP_VSA8", "rd_EState_VSA4", "rd_EState_VSA5", "rd_EState_VSA7", "rd_EState_VSA8", "rd_VSA_EState3", "rd_VSA_EState4", "rd_VSA_EState6", "rd_VSA_EState7", "rd_VSA_EState8", "rd_VSA_EState9", "rd_NumAliphaticCarbocycles", "rd_NumAliphaticHeterocycles", "rd_NumAliphaticRings", "rd_NumAromaticCarbocycles", "rd_NumAromaticRings", "rd_NumHeterocycles", "rd_NumRotatableBonds", "rd_NumSaturatedHeterocycles", "rd_NumSaturatedRings", "rd_RingCount", "rd_MolLogP", "rd_fr_Ar_OH", "rd_fr_NH0", "rd_fr_Ndealkylation2", "rd_fr_bicyclic", "rd_fr_piperdine", "rd_fr_piperzine", "rd_fr_unbrch_alkane"]
    ),
    ViabilityEstimator( 
        date='2c71181e-2776-11f1-9087-60cf848a7bd6',
        device=DEVICE,
        model='tabpfn',
        model_type='regressor',
        run_num=5,
        features=['rd_MaxAbsEStateIndex', 'rd_qed', 'rd_SPS', 'rd_MolWt', 'rd_NumValenceElectrons', 'rd_MaxPartialCharge', 'rd_MinAbsPartialCharge', 'rd_FpDensityMorgan1', 'rd_FpDensityMorgan2', 'rd_BCUT2D_MWHI', 'rd_BCUT2D_MWLOW', 'rd_BCUT2D_CHGHI', 'rd_BCUT2D_LOGPLOW', 'rd_BCUT2D_MRHI', 'rd_AvgIpc', 'rd_BalabanJ', 'rd_BertzCT', 'rd_Chi1v', 'rd_Chi2n', 'rd_Chi2v', 'rd_Chi3v', 'rd_Chi4n', 'rd_Ipc', 'rd_Kappa2', 'rd_Kappa3', 'rd_PEOE_VSA11', 'rd_PEOE_VSA12', 'rd_PEOE_VSA13', 'rd_PEOE_VSA14', 'rd_PEOE_VSA2', 'rd_PEOE_VSA3', 'rd_PEOE_VSA4', 'rd_PEOE_VSA6', 'rd_PEOE_VSA7', 'rd_PEOE_VSA8', 'rd_SMR_VSA10', 'rd_SMR_VSA2', 'rd_SMR_VSA3', 'rd_SMR_VSA4', 'rd_SMR_VSA5', 'rd_SMR_VSA6', 'rd_SMR_VSA7', 'rd_SlogP_VSA1', 'rd_SlogP_VSA10', 'rd_SlogP_VSA12', 'rd_SlogP_VSA2', 'rd_SlogP_VSA5', 'rd_SlogP_VSA6', 'rd_SlogP_VSA7', 'rd_TPSA', 'rd_EState_VSA10', 'rd_EState_VSA4', 'rd_EState_VSA5', 'rd_EState_VSA7', 'rd_EState_VSA8', 'rd_VSA_EState10', 'rd_VSA_EState4', 'rd_VSA_EState6', 'rd_VSA_EState7', 'rd_VSA_EState8', 'rd_FractionCSP3', 'rd_NHOHCount', 'rd_NumAliphaticHeterocycles', 'rd_NumAliphaticRings', 'rd_NumAmideBonds', 'rd_NumAromaticCarbocycles', 'rd_NumAromaticRings', 'rd_NumAtomStereoCenters', 'rd_NumBridgeheadAtoms', 'rd_NumHAcceptors', 'rd_NumRotatableBonds', 'rd_NumSaturatedHeterocycles', 'rd_NumSaturatedRings', 'rd_NumUnspecifiedAtomStereoCenters', 'rd_RingCount', 'rd_MolLogP', 'rd_fr_Al_COO', 'rd_fr_ArN', 'rd_fr_Ar_NH', 'rd_fr_NH2', 'rd_fr_Ndealkylation1', 'rd_fr_Ndealkylation2', 'rd_fr_amidine', 'rd_fr_azo', 'rd_fr_halogen', 'rd_fr_imidazole', 'rd_fr_ketone', 'rd_fr_ketone_Topliss', 'rd_fr_methoxy', 'rd_fr_nitrile', 'rd_fr_nitro', 'rd_fr_para_hydroxylation', 'rd_fr_piperdine', 'rd_fr_piperzine', 'rd_fr_sulfide', 'rd_fr_thiazole', 'rd_fr_unbrch_alkane', 'rd_fr_urea']
    ),
    ViabilityEstimator( 
        date='6263e416-277a-11f1-9087-60cf848a7bd6',
        device=DEVICE,
        model='tabpfn',
        model_type='regressor',
        run_num=5,
        features=['rd_MaxAbsEStateIndex', 'rd_MinAbsEStateIndex', 'rd_qed', 'rd_SPS', 'rd_MolWt', 'rd_NumValenceElectrons', 'rd_MinPartialCharge', 'rd_MaxAbsPartialCharge', 'rd_FpDensityMorgan3', 'rd_BCUT2D_MWHI', 'rd_BCUT2D_MWLOW', 'rd_BCUT2D_CHGHI', 'rd_BCUT2D_MRHI', 'rd_BCUT2D_MRLOW', 'rd_AvgIpc', 'rd_BalabanJ', 'rd_BertzCT', 'rd_Chi1v', 'rd_Chi2n', 'rd_Chi2v', 'rd_Chi3v', 'rd_Chi4n', 'rd_HallKierAlpha', 'rd_Ipc', 'rd_Kappa2', 'rd_PEOE_VSA11', 'rd_PEOE_VSA13', 'rd_PEOE_VSA5', 'rd_PEOE_VSA6', 'rd_PEOE_VSA7', 'rd_PEOE_VSA9', 'rd_SMR_VSA10', 'rd_SMR_VSA2', 'rd_SMR_VSA5', 'rd_SMR_VSA6', 'rd_SMR_VSA7', 'rd_SMR_VSA9', 'rd_SlogP_VSA1', 'rd_SlogP_VSA11', 'rd_SlogP_VSA12', 'rd_SlogP_VSA2', 'rd_SlogP_VSA4', 'rd_SlogP_VSA5', 'rd_SlogP_VSA6', 'rd_SlogP_VSA7', 'rd_TPSA', 'rd_EState_VSA1', 'rd_EState_VSA3', 'rd_EState_VSA4', 'rd_EState_VSA6', 'rd_EState_VSA7', 'rd_EState_VSA8', 'rd_VSA_EState10', 'rd_VSA_EState4', 'rd_VSA_EState5', 'rd_VSA_EState6', 'rd_VSA_EState7', 'rd_VSA_EState8', 'rd_VSA_EState9', 'rd_NHOHCount', 'rd_NumAliphaticCarbocycles', 'rd_NumAliphaticHeterocycles', 'rd_NumAromaticCarbocycles', 'rd_NumAromaticRings', 'rd_NumAtomStereoCenters', 'rd_NumBridgeheadAtoms', 'rd_NumHAcceptors', 'rd_NumHeteroatoms', 'rd_NumHeterocycles', 'rd_NumSaturatedCarbocycles', 'rd_NumSaturatedRings', 'rd_NumUnspecifiedAtomStereoCenters', 'rd_RingCount', 'rd_MolLogP', 'rd_fr_Al_OH', 'rd_fr_ArN', 'rd_fr_Ar_OH', 'rd_fr_C_S', 'rd_fr_NH0', 'rd_fr_NH1', 'rd_fr_NH2', 'rd_fr_SH', 'rd_fr_alkyl_halide', 'rd_fr_allylic_oxid', 'rd_fr_aryl_methyl', 'rd_fr_azo', 'rd_fr_guanido', 'rd_fr_halogen', 'rd_fr_imide', 'rd_fr_lactone', 'rd_fr_nitro', 'rd_fr_para_hydroxylation', 'rd_fr_quatN', 'rd_fr_term_acetylene', 'rd_fr_thiocyan', 'rd_fr_unbrch_alkane']
    ),
    ViabilityEstimator( 
        date='eff9653e-277b-11f1-9087-60cf848a7bd6',
        device=DEVICE,
        model='tabpfn',
        model_type='regressor',
        run_num=5,
        features=['rd_MaxAbsEStateIndex', 'rd_MinEStateIndex', 'rd_qed', 'rd_SPS', 'rd_MolWt', 'rd_NumValenceElectrons', 'rd_FpDensityMorgan1', 'rd_FpDensityMorgan2', 'rd_BCUT2D_MWHI', 'rd_BCUT2D_MWLOW', 'rd_BCUT2D_CHGHI', 'rd_BCUT2D_CHGLO', 'rd_BCUT2D_LOGPHI', 'rd_BCUT2D_MRHI', 'rd_AvgIpc', 'rd_BalabanJ', 'rd_BertzCT', 'rd_Chi1v', 'rd_Chi2n', 'rd_Chi3v', 'rd_Chi4n', 'rd_HallKierAlpha', 'rd_Ipc', 'rd_Kappa2', 'rd_Kappa3', 'rd_PEOE_VSA1', 'rd_PEOE_VSA10', 'rd_PEOE_VSA12', 'rd_PEOE_VSA13', 'rd_PEOE_VSA14', 'rd_PEOE_VSA2', 'rd_PEOE_VSA3', 'rd_PEOE_VSA6', 'rd_PEOE_VSA7', 'rd_SMR_VSA10', 'rd_SMR_VSA3', 'rd_SMR_VSA4', 'rd_SMR_VSA5', 'rd_SMR_VSA6', 'rd_SMR_VSA7', 'rd_SMR_VSA9', 'rd_SlogP_VSA1', 'rd_SlogP_VSA10', 'rd_SlogP_VSA11', 'rd_SlogP_VSA12', 'rd_SlogP_VSA2', 'rd_SlogP_VSA3', 'rd_SlogP_VSA4', 'rd_SlogP_VSA5', 'rd_SlogP_VSA6', 'rd_SlogP_VSA7', 'rd_TPSA', 'rd_EState_VSA3', 'rd_EState_VSA4', 'rd_EState_VSA5', 'rd_EState_VSA7', 'rd_EState_VSA8', 'rd_VSA_EState10', 'rd_VSA_EState2', 'rd_VSA_EState5', 'rd_VSA_EState6', 'rd_VSA_EState7', 'rd_VSA_EState8', 'rd_FractionCSP3', 'rd_NHOHCount', 'rd_NumAliphaticCarbocycles', 'rd_NumAliphaticRings', 'rd_NumAromaticCarbocycles', 'rd_NumAromaticHeterocycles', 'rd_NumAromaticRings', 'rd_NumAtomStereoCenters', 'rd_NumBridgeheadAtoms', 'rd_NumHAcceptors', 'rd_NumHeterocycles', 'rd_NumRotatableBonds', 'rd_NumSaturatedCarbocycles', 'rd_NumSaturatedRings', 'rd_NumSpiroAtoms', 'rd_RingCount', 'rd_MolLogP', 'rd_fr_ArN', 'rd_fr_Ar_N', 'rd_fr_Ar_NH', 'rd_fr_C_S', 'rd_fr_NH0', 'rd_fr_NH1', 'rd_fr_NH2', 'rd_fr_SH', 'rd_fr_aldehyde', 'rd_fr_alkyl_halide', 'rd_fr_allylic_oxid', 'rd_fr_aniline', 'rd_fr_bicyclic', 'rd_fr_ester', 'rd_fr_ether', 'rd_fr_guanido', 'rd_fr_halogen', 'rd_fr_imidazole', 'rd_fr_nitro', 'rd_fr_nitro_arom_nonortho', 'rd_fr_phenol', 'rd_fr_priamide', 'rd_fr_quatN', 'rd_fr_sulfonamd', 'rd_fr_thiocyan']
    ),
    ViabilityEstimator( 
        date='53a80028-277b-11f1-9087-60cf848a7bd6',
        device=DEVICE,
        model='tabpfn',
        model_type='regressor',
        run_num=5,
        features=['rd_MinAbsEStateIndex', 'rd_qed', 'rd_SPS', 'rd_MolWt', 'rd_NumValenceElectrons', 'rd_MinPartialCharge', 'rd_BCUT2D_MWHI', 'rd_BCUT2D_MWLOW', 'rd_BCUT2D_CHGHI', 'rd_BCUT2D_CHGLO', 'rd_BCUT2D_LOGPHI', 'rd_BCUT2D_MRLOW', 'rd_AvgIpc', 'rd_BalabanJ', 'rd_Chi1v', 'rd_Chi2n', 'rd_Chi3v', 'rd_Chi4n', 'rd_HallKierAlpha', 'rd_Ipc', 'rd_Kappa2', 'rd_Kappa3', 'rd_PEOE_VSA1', 'rd_PEOE_VSA10', 'rd_PEOE_VSA11', 'rd_PEOE_VSA12', 'rd_PEOE_VSA13', 'rd_PEOE_VSA2', 'rd_PEOE_VSA6', 'rd_PEOE_VSA7', 'rd_PEOE_VSA9', 'rd_SMR_VSA1', 'rd_SMR_VSA10', 'rd_SMR_VSA5', 'rd_SMR_VSA7', 'rd_SlogP_VSA1', 'rd_SlogP_VSA10', 'rd_SlogP_VSA11', 'rd_SlogP_VSA12', 'rd_SlogP_VSA2', 'rd_SlogP_VSA3', 'rd_SlogP_VSA4', 'rd_SlogP_VSA5', 'rd_SlogP_VSA6', 'rd_TPSA', 'rd_EState_VSA1', 'rd_EState_VSA10', 'rd_EState_VSA2', 'rd_EState_VSA3', 'rd_EState_VSA4', 'rd_EState_VSA7', 'rd_EState_VSA8', 'rd_VSA_EState10', 'rd_VSA_EState2', 'rd_VSA_EState3', 'rd_VSA_EState5', 'rd_VSA_EState6', 'rd_VSA_EState7', 'rd_VSA_EState8', 'rd_FractionCSP3', 'rd_NHOHCount', 'rd_NumAliphaticHeterocycles', 'rd_NumAmideBonds', 'rd_NumAromaticHeterocycles', 'rd_NumAtomStereoCenters', 'rd_NumHAcceptors', 'rd_NumHeteroatoms', 'rd_NumHeterocycles', 'rd_NumRotatableBonds', 'rd_NumSaturatedCarbocycles', 'rd_NumSaturatedHeterocycles', 'rd_NumSaturatedRings', 'rd_NumUnspecifiedAtomStereoCenters', 'rd_MolLogP', 'rd_fr_Al_OH_noTert', 'rd_fr_ArN', 'rd_fr_Ar_N', 'rd_fr_Ar_OH', 'rd_fr_NH0', 'rd_fr_NH1', 'rd_fr_NH2', 'rd_fr_Ndealkylation2', 'rd_fr_alkyl_halide', 'rd_fr_allylic_oxid', 'rd_fr_aniline', 'rd_fr_bicyclic', 'rd_fr_ester', 'rd_fr_ether', 'rd_fr_guanido', 'rd_fr_halogen', 'rd_fr_hdrzone', 'rd_fr_ketone', 'rd_fr_lactone', 'rd_fr_nitro', 'rd_fr_nitro_arom_nonortho', 'rd_fr_para_hydroxylation', 'rd_fr_sulfide', 'rd_fr_thiazole']
    ),
    ViabilityEstimator( 
        date='108f5a82-277d-11f1-9087-60cf848a7bd6',
        device=DEVICE,
        model='tabpfn',
        model_type='regressor',
        run_num=5,
        features=['rd_MaxAbsEStateIndex', 'rd_MinAbsEStateIndex', 'rd_MinEStateIndex', 'rd_qed', 'rd_SPS', 'rd_MolWt', 'rd_MaxPartialCharge', 'rd_MinPartialCharge', 'rd_FpDensityMorgan1', 'rd_FpDensityMorgan2', 'rd_BCUT2D_MWHI', 'rd_BCUT2D_MWLOW', 'rd_BCUT2D_CHGHI', 'rd_BCUT2D_MRHI', 'rd_BCUT2D_MRLOW', 'rd_AvgIpc', 'rd_BalabanJ', 'rd_BertzCT', 'rd_Chi0n', 'rd_Chi1v', 'rd_Chi2v', 'rd_Chi3n', 'rd_Chi3v', 'rd_Chi4v', 'rd_HallKierAlpha', 'rd_Ipc', 'rd_Kappa2', 'rd_Kappa3', 'rd_PEOE_VSA12', 'rd_PEOE_VSA14', 'rd_PEOE_VSA3', 'rd_PEOE_VSA6', 'rd_PEOE_VSA7', 'rd_PEOE_VSA8', 'rd_PEOE_VSA9', 'rd_SMR_VSA1', 'rd_SMR_VSA10', 'rd_SMR_VSA2', 'rd_SMR_VSA5', 'rd_SMR_VSA7', 'rd_SMR_VSA9', 'rd_SlogP_VSA1', 'rd_SlogP_VSA11', 'rd_SlogP_VSA2', 'rd_SlogP_VSA4', 'rd_SlogP_VSA5', 'rd_SlogP_VSA6', 'rd_SlogP_VSA8', 'rd_TPSA', 'rd_EState_VSA1', 'rd_EState_VSA10', 'rd_EState_VSA11', 'rd_EState_VSA2', 'rd_EState_VSA3', 'rd_EState_VSA4', 'rd_EState_VSA5', 'rd_EState_VSA6', 'rd_EState_VSA7', 'rd_EState_VSA8', 'rd_EState_VSA9', 'rd_VSA_EState3', 'rd_VSA_EState4', 'rd_VSA_EState6', 'rd_VSA_EState7', 'rd_VSA_EState8', 'rd_VSA_EState9', 'rd_NOCount', 'rd_NumAliphaticCarbocycles', 'rd_NumAliphaticHeterocycles', 'rd_NumAliphaticRings', 'rd_NumAromaticCarbocycles', 'rd_NumAromaticRings', 'rd_NumAtomStereoCenters', 'rd_NumBridgeheadAtoms', 'rd_NumRotatableBonds', 'rd_NumSaturatedCarbocycles', 'rd_NumSaturatedHeterocycles', 'rd_NumSaturatedRings', 'rd_NumSpiroAtoms', 'rd_RingCount', 'rd_MolLogP', 'rd_fr_ArN', 'rd_fr_Ar_N', 'rd_fr_Ar_OH', 'rd_fr_C_O', 'rd_fr_C_S', 'rd_fr_Imine', 'rd_fr_NH0', 'rd_fr_NH1', 'rd_fr_NH2', 'rd_fr_N_O', 'rd_fr_Ndealkylation1', 'rd_fr_Ndealkylation2', 'rd_fr_SH', 'rd_fr_alkyl_carbamate', 'rd_fr_allylic_oxid', 'rd_fr_amidine', 'rd_fr_aryl_methyl', 'rd_fr_azo', 'rd_fr_bicyclic', 'rd_fr_epoxide', 'rd_fr_halogen', 'rd_fr_hdrzone', 'rd_fr_imidazole', 'rd_fr_nitrile', 'rd_fr_nitro', 'rd_fr_nitro_arom', 'rd_fr_nitro_arom_nonortho', 'rd_fr_oxime', 'rd_fr_para_hydroxylation', 'rd_fr_piperdine', 'rd_fr_quatN', 'rd_fr_thiocyan']
    ),
    ViabilityEstimator( 
        date='7c617e02-277d-11f1-9087-60cf848a7bd6',
        device=DEVICE,
        model='tabpfn',
        model_type='regressor',
        run_num=5,
        features=['rd_MinAbsEStateIndex', 'rd_MinEStateIndex', 'rd_qed', 'rd_SPS', 'rd_MaxPartialCharge', 'rd_MinPartialCharge', 'rd_FpDensityMorgan3', 'rd_BCUT2D_MWHI', 'rd_BCUT2D_MWLOW', 'rd_BCUT2D_CHGHI', 'rd_BCUT2D_MRHI', 'rd_BCUT2D_MRLOW', 'rd_BalabanJ', 'rd_Chi0n', 'rd_Chi2v', 'rd_Chi3n', 'rd_Chi4v', 'rd_HallKierAlpha', 'rd_PEOE_VSA1', 'rd_PEOE_VSA11', 'rd_PEOE_VSA12', 'rd_PEOE_VSA13', 'rd_PEOE_VSA14', 'rd_PEOE_VSA2', 'rd_PEOE_VSA3', 'rd_PEOE_VSA4', 'rd_PEOE_VSA5', 'rd_PEOE_VSA6', 'rd_PEOE_VSA7', 'rd_PEOE_VSA8', 'rd_PEOE_VSA9', 'rd_SMR_VSA1', 'rd_SMR_VSA10', 'rd_SMR_VSA2', 'rd_SMR_VSA3', 'rd_SMR_VSA5', 'rd_SMR_VSA6', 'rd_SMR_VSA7', 'rd_SMR_VSA9', 'rd_SlogP_VSA1', 'rd_SlogP_VSA10', 'rd_SlogP_VSA11', 'rd_SlogP_VSA12', 'rd_SlogP_VSA2', 'rd_SlogP_VSA4', 'rd_SlogP_VSA5', 'rd_SlogP_VSA6', 'rd_SlogP_VSA7', 'rd_TPSA', 'rd_EState_VSA1', 'rd_EState_VSA10', 'rd_EState_VSA11', 'rd_EState_VSA2', 'rd_EState_VSA3', 'rd_EState_VSA4', 'rd_EState_VSA5', 'rd_EState_VSA7', 'rd_EState_VSA8', 'rd_EState_VSA9', 'rd_VSA_EState1', 'rd_VSA_EState10', 'rd_VSA_EState2', 'rd_VSA_EState4', 'rd_VSA_EState5', 'rd_VSA_EState6', 'rd_VSA_EState7', 'rd_VSA_EState8', 'rd_VSA_EState9', 'rd_FractionCSP3', 'rd_NHOHCount', 'rd_NOCount', 'rd_NumAliphaticHeterocycles', 'rd_NumAmideBonds', 'rd_NumAromaticCarbocycles', 'rd_NumAromaticHeterocycles', 'rd_NumAromaticRings', 'rd_NumBridgeheadAtoms', 'rd_NumHeteroatoms', 'rd_NumHeterocycles', 'rd_NumSaturatedCarbocycles', 'rd_NumSaturatedHeterocycles', 'rd_NumUnspecifiedAtomStereoCenters', 'rd_RingCount', 'rd_MolLogP', 'rd_fr_Al_OH', 'rd_fr_Al_OH_noTert', 'rd_fr_ArN', 'rd_fr_Ar_N', 'rd_fr_Ar_NH', 'rd_fr_Ar_OH', 'rd_fr_C_O', 'rd_fr_C_S', 'rd_fr_HOCCN', 'rd_fr_NH0', 'rd_fr_NH1', 'rd_fr_NH2', 'rd_fr_N_O', 'rd_fr_Ndealkylation1', 'rd_fr_Ndealkylation2', 'rd_fr_SH', 'rd_fr_aldehyde', 'rd_fr_allylic_oxid', 'rd_fr_amidine', 'rd_fr_aniline', 'rd_fr_aryl_methyl', 'rd_fr_azo', 'rd_fr_bicyclic', 'rd_fr_epoxide', 'rd_fr_ether', 'rd_fr_guanido', 'rd_fr_halogen', 'rd_fr_hdrzone', 'rd_fr_ketone_Topliss', 'rd_fr_lactone', 'rd_fr_methoxy', 'rd_fr_nitrile', 'rd_fr_nitro', 'rd_fr_oxazole', 'rd_fr_para_hydroxylation', 'rd_fr_phos_acid', 'rd_fr_pyridine', 'rd_fr_quatN', 'rd_fr_sulfone', 'rd_fr_term_acetylene', 'rd_fr_thiazole', 'rd_fr_thiophene', 'rd_fr_unbrch_alkane']
    ),

]

In [ ]:
# Predict

# init
shape = (len(estimator_list), input_data.shape[0])
result_array = np.zeros(shape, dtype=np.float64)

for idx in tqdm.trange(len(estimator_list)):
    estimator = estimator_list[idx]
    if estimator.model_type == "classifier":
        # select the best probabiliy based on results
        probas = estimator.predict_proba(input_data)
        pred = np.argmax(probas, axis=1)
        result_array[idx] = pred
    else:
        # directly return pred
        pred = estimator.predict(input_data)
        result_array[idx] = pred

In [ ]:
# Results
def num_to_label(X, CAT_KEYS):
    cat_keys_array = np.asarray(CAT_KEYS)
    index_array = np.astype(X, np.int64)
    return cat_keys_array[index_array]

# turn numbric categories to labels
data = np.zeros_like(result_array, dtype=object)
for i in range(result_array.shape[0]):
    estimator = estimator_list[i]
    if estimator.model_type == "classifier":
        cat_keys = estimator.cat_keys
        data[i] = num_to_label(result_array[i], cat_keys)
    else:
        data[i] = result_array[i]

# dataframe
result_df = pd.DataFrame(data.T, columns=[i.date for i in estimator_list])

# rename for better readability
result_df = result_df.rename(columns=MODEL_MAPPINGS)

# merge
smiles_series = input_data['smiles']
smi_duplicated = smiles_series.duplicated().rename('is_duplicated')
result_df = pd.concat(
    [smiles_series, smi_duplicated, result_df],
    axis=1,
)

In [ ]:
# export results
result_df.to_csv(join("data_output.csv"), index=False)